# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAliEjaz1/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which declining search pages should a content editor
review first, given limited weekly review time?

**Decision supported:** prioritizing a finite editorial review queue.
**Unit of analysis:** one page's monthly performance snapshot.
**Cost of a wrong call:** flagging a healthy page wastes review time; missing
a genuinely declining, visible page lets it lose ground silently — the more
expensive mistake.

In [9]:
question = "Which declining search pages should a content editor review first?"
decision = "prioritizing a finite editorial review queue"
print(f"Question: {question}")
print(f"Decision: {decision}")

Question: Which declining search pages should a content editor review first?
Decision: prioritizing a finite editorial review queue


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank ML Internship warehouse (Hugging Face), pages and
clients identified only by hashed IDs.
**Tables used:** `fact_content_daily_performance` (monthly slices, Jan–Jun
2026), aggregated to one row per page per client per month.
**Date windows:** train Jan–Feb 2026, test Apr–May 2026; the sealed final
month (June 2026 / `_sample` table) was never used.
**Excluded:** raw query text, raw URLs, any future-window metric that could
leak the outcome into the features.

In [10]:
release = "FlyRank ML Internship warehouse (Hugging Face)"
table = "fact_content_daily_performance"
train_window = "2026-01 to 2026-02"
test_window = "2026-04 to 2026-05"
excluded = ["raw query text", "raw URLs", "future-window metrics"]
print(f"Release: {release}\nTable: {table}")
print(f"Train: {train_window}, Test: {test_window}")
print(f"Excluded: {excluded}")

Release: FlyRank ML Internship warehouse (Hugging Face)
Table: fact_content_daily_performance
Train: 2026-01 to 2026-02, Test: 2026-04 to 2026-05
Excluded: ['raw query text', 'raw URLs', 'future-window metrics']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Features:** average search position, total impressions, total clicks,
click-through rate — all knowable at decision time.
**Label:** declining = clicks next month < 80% of this month's clicks, among
pages with ≥100 impressions. Known artifact: zero-click pages (39.6% of the
panel) can never be labeled declining by this formula.
**Baseline:** hand rule flagging positions 11–20 with real impressions, plus
a secondary flag for top-10 pages with low CTR.
**Model:** decision tree, max depth 4, class-balanced.
**Validation:** time-aware (train before test) AND grouped by client (30% of
clients held out entirely from training) — both were necessary; page overlap
between months alone was 52,792 pages before grouping.
**Leakage checks:** confirmed no feature depends on future data; confirmed
the zero-click label artifact; confirmed train/test client sets don't overlap.

In [11]:
features = ["avg_position", "total_impressions", "total_clicks", "ctr"]
label_rule = "clicks_next < 0.8 * total_clicks, among pages with impressions >= 100"
baseline_rule = "position 11-20 with impressions>=100, or position<=10 with low CTR"
model = "DecisionTreeClassifier(max_depth=4, class_weight='balanced')"
validation = "time-aware split + grouped by client (30% clients held out)"
print(f"Features: {features}")
print(f"Label: {label_rule}")
print(f"Baseline: {baseline_rule}")
print(f"Model: {model}")
print(f"Validation: {validation}")

Features: ['avg_position', 'total_impressions', 'total_clicks', 'ctr']
Label: clicks_next < 0.8 * total_clicks, among pages with impressions >= 100
Baseline: position 11-20 with impressions>=100, or position<=10 with low CTR
Model: DecisionTreeClassifier(max_depth=4, class_weight='balanced')
Validation: time-aware split + grouped by client (30% clients held out)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Mean P@50 across 10 client-held-out draws: tree 0.60 (range 0.26–0.80,
won 9 of 10 draws) vs rule 0.30 (range 0.24–0.36). On eligible pages only
(clicks>0, removing the label artifact): tree 0.64 vs rule 0.44. The tree's
top 500 and the rule's top 500 share zero pages — the two methods find
different populations, not overlapping ones.

In [12]:
import pandas as pd
results = pd.DataFrame({
    "method": ["Week 4 rule", "Decision tree"],
    "mean_P@50_10_draws": [0.30, 0.60],
    "range": ["0.24-0.36", "0.26-0.80"],
    "draws_won": ["1 of 10", "9 of 10"],
    "P@50_eligible_pages_only": [0.44, 0.64],
})
print(results.to_string(index=False))
print("\nTree top-500 vs rule top-500 overlap: 0 pages")

       method  mean_P@50_10_draws     range draws_won  P@50_eligible_pages_only
  Week 4 rule                 0.3 0.24-0.36   1 of 10                      0.44
Decision tree                 0.6 0.26-0.80   9 of 10                      0.64

Tree top-500 vs rule top-500 overlap: 0 pages


## 5. Limitations

*What this work cannot claim.*

- Small client panel (50 clients, 15 held out per draw) — directional, not
  production-grade.
- Result instability — tree's P@50 ranged 0.26 to 0.80 depending on which
  clients were tested.
- Label artifact — 39.6% of pages are permanently ineligible to be labeled
  declining.
- No causal claim — nothing here shows a signal causes decline, or that
  refreshing recovers clicks. Evidence is observational and directional.

In [13]:
limitations = [
    "small client panel (n=50)",
    "P@50 instability across client draws (0.26-0.80)",
    "label artifact: 39.6% of pages can never be labeled declining",
    "observational only, no causal claim",
]
for l in limitations:
    print(f"- {l}")

- small client panel (n=50)
- P@50 instability across client draws (0.26-0.80)
- label artifact: 39.6% of pages can never be labeled declining
- observational only, no causal claim


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Because the tree and rule flag disjoint pages, the playbook treats them as
two separate signals, not a blended score. `rule_only` and `tree_only` pages
go to review; `neither_flag` pages get no action. Never automate publishing
or removal decisions from the score alone. Retrain if fresh-month precision
drops well below 0.60, or after ~90 days.

In [14]:
playbook = {
    "rule_only": "review_for_refresh",
    "tree_only": "review_for_refresh (lower confidence)",
    "neither_flag": "no_action",
}
never_automate = ["auto-publish changes", "auto-remove/deprioritize a page",
                   "treat tree_only as equal confidence to manual review"]
for k, v in playbook.items():
    print(f"{k}: {v}")
print("Never automate:", never_automate)

rule_only: review_for_refresh
tree_only: review_for_refresh (lower confidence)
neither_flag: no_action
Never automate: ['auto-publish changes', 'auto-remove/deprioritize a page', 'treat tree_only as equal confidence to manual review']


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

These are the exact numbers embedded as charts/tables in the deployed paper
(muhammadaliejaz1.github.io/flyrank-ml-internship-starter/) — collected here
so the paper's figures trace back to committed code.

In [15]:
import json
paper_artifacts = {
    "baseline_mean_P50": 0.30,
    "tree_mean_P50": 0.60,
    "tree_P50_range": [0.26, 0.80],
    "tree_P50_eligible_only": 0.64,
    "baseline_P50_eligible_only": 0.44,
    "tree_rule_top500_overlap": 0,
    "zero_click_page_share": 0.396,
    "clients_evaluated": 50,
}
print(json.dumps(paper_artifacts, indent=2))

{
  "baseline_mean_P50": 0.3,
  "tree_mean_P50": 0.6,
  "tree_P50_range": [
    0.26,
    0.8
  ],
  "tree_P50_eligible_only": 0.64,
  "baseline_P50_eligible_only": 0.44,
  "tree_rule_top500_overlap": 0,
  "zero_click_page_share": 0.396,
  "clients_evaluated": 50
}


**5-minute demo outline:**
1. The question (30s) — which declining pages to review first
2. The baseline rule and why it's a fair bar (1 min)
3. The tree's result and the honest audit — 0.60 mean, but 0.26-0.80 range (1.5 min)
4. The zero-overlap finding — the two methods disagree entirely (1 min)
5. The playbook and no-go list (1 min)

**Social-post cut (2-3 sentences):**
"Built a page-refresh prioritization model on 79M rows of real search data
for my FlyRank ML internship. The headline number looked great until I
audited it on unseen clients — the honest result is more interesting than
the clean one. Full paper + code: [repo link]"

**3-sentence employer-facing summary:**
"Built and audited a decision-tree scorer for prioritizing content refresh
decisions on 79M rows of production search data. A client-grouped validation
audit revealed the model's advantage over a hand-written baseline was real
but unstable (0.60 mean precision@50, ranging 0.26-0.80), and that the model
and baseline flagged entirely disjoint pages — a finding that reshaped the
final recommendation into a two-signal playbook rather than a single score."

In [16]:
print("Demo outline, social post, and employer summary drafted above in markdown.")

Demo outline, social post, and employer summary drafted above in markdown.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
